# Paywall A/B Test — Statistical Significance

Compute a two-proportion z-test for the new paywall A/B test using data from the `ab_tests.ab_tests` table.

In [ ]:
import pandas as pd
from google.cloud import bigquery
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

PROJECT = 'hopeful-list-429812-f3'
TEST_NAME = 'paywall_v2_test'
DATE_FROM, DATE_TO = '2026-04-01', '2026-04-21'

client = bigquery.Client(project=PROJECT)

In [ ]:
query = f"""
SELECT
  test_version,
  COUNTIF(metric_name = 'paywall_view') AS exposures,
  COUNTIF(metric_name = 'trial_start')  AS conversions
FROM `{PROJECT}.ab_tests.ab_tests`
WHERE test_name = '{TEST_NAME}'
  AND DATE(date) BETWEEN '{DATE_FROM}' AND '{DATE_TO}'
GROUP BY test_version
"""
df = client.query(query).to_dataframe()
df

In [ ]:
control = df[df.test_version == 'control'].iloc[0]
variant = df[df.test_version == 'variant'].iloc[0]

successes = [variant.conversions, control.conversions]
trials    = [variant.exposures,   control.exposures]

z_stat, p_value = proportions_ztest(successes, trials, alternative='larger')

cr_control = control.conversions / control.exposures
cr_variant = variant.conversions / variant.exposures
uplift = cr_variant - cr_control
rel_uplift = uplift / cr_control

ci_low, ci_high = proportion_confint(variant.conversions, variant.exposures, alpha=0.05, method='wilson')

print(f'Control CR: {cr_control:.4f}')
print(f'Variant CR: {cr_variant:.4f}  (95% CI: {ci_low:.4f} .. {ci_high:.4f})')
print(f'Absolute uplift: {uplift:+.4f}')
print(f'Relative uplift: {rel_uplift:+.2%}')
print(f'z = {z_stat:.3f}, p-value = {p_value:.4f}')

## Decision

If `p_value < 0.05` and the relative uplift is meaningfully positive, we recommend rolling out the variant.

Caveats:
- Only one primary metric is tested here (trial_start). Long-term retention should be re-measured 30 days post rollout.
- The test should run for at least one full weekly cycle to absorb day-of-week effects.